# Step 5 (Apply) + Step 7 — Calibrated Config & Unified Deployment Feed

**Part A** applies the calibration notebook's findings to concrete config changes -- including
one structural fix the sweep data itself points to, not just new threshold numbers. **Part B**
is Step 7: merges Tier 1 (always-on) and Tier 2 (conditional) outputs into the single unified
schema every downstream consumer reads, and produces the actual "Disruption Alerts" export.

**Decisions made in Part A, and why (grounded in your real calibration output, not assumed):**
- `COMPOSITE_TRIGGER_THRESHOLD` stays at **0.65** -- the sweep shows this parameter barely moves
  anything across 0.55-0.75 (`n_composite_driven` is 0-7 almost everywhere). It isn't the lever.
- `SUBSCORE_DISAGREEMENT_THRESHOLD` moves **0.35 -> 0.45**, *and* the disagreement calculation
  now excludes `fundamentals_subscore`. Reasoning: even at the top of the tested grid (0.50) the
  trigger rate barely improved (33.1% -> 23.4%) -- that flatness points to a structural cause,
  not a mistuned number. Fundamentals is monthly and forward-filled (the same value repeats for
  weeks) while news/vessel move daily; comparing a stale value against volatile ones will look
  like "disagreement" mechanically, regardless of real risk. Excluding it from the spread (it's
  still used in the composite score itself, just not in the disagreement check) targets the
  actual cause instead of just raising a threshold against it.
- `HUMAN_REVIEW_DISAGREEMENT_THRESHOLD` moves **0.30 -> 0.20** -- your own live 20-episode run
  makes this one unambiguous: the flat 0.30 threshold flagged **zero** episodes, while the real
  `disagreement_index` distribution topped out at 0.205 at the 95th percentile. The old value
  was set above the ceiling of what the metric could ever produce.
- `HIGH_CRED_SCORE_FLOOR` is **left at 0.30, unchanged** -- the calibration notebook doesn't
  cover it at all. Flagging this explicitly rather than pretending it's been validated.

**Part B's scope, and what's deliberately left out:** merges Tier 1 + Tier 2 into one unified
per-corridor-timestamp table and produces a ranked Disruption Alerts export. It does **not**
build a Knowledge Graph, Procurement Orchestrator, or Reserve Optimisation Agent -- none of
those have real supplier/refinery/route data anywhere in this project, and fabricating them
without real inputs would be worse than not building them. This notebook stops at the point
those downstream agents would actually consume real output from.

---
# Part A — Apply Calibration

## 0. Config

In [1]:
import json
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

# --- INPUTS (Step 5's outputs) ---
OUTPUT_DIR = Path("outputs")
TIER1_COMPOSITE_PATH = Path(r"D:\python time\ET\code\risk_agent\code\outputs\tier1_composite.csv")
SWEEP_PATH = OUTPUT_DIR / "step5_threshold_sweep.csv"
CALIBRATED_CONFIG_PATH = OUTPUT_DIR / "step5_calibrated_config.json"
STEP4_EPISODES_PATH = OUTPUT_DIR / "step4_trigger_episodes.csv"
STEP4_FINAL_RESULTS_PATH = OUTPUT_DIR / "step4_final_risk_scores.json"

# --- OUTPUT ---
FINAL_CALIBRATED_CONFIG = OUTPUT_DIR / "step5_final_calibrated_config.json"
UNIFIED_FEED_OUTPUT = OUTPUT_DIR / "unified_risk_feed.csv"
DISRUPTION_ALERTS_OUTPUT = OUTPUT_DIR / "disruption_alerts.csv"

CORRIDORS = ["hormuz", "bab_el_mandeb", "suez", "red_sea", "persian_gulf", "india"]

# --- CALIBRATED VALUES (see the intro cell for the reasoning behind each) ---
NEW_COMPOSITE_TRIGGER_THRESHOLD = 0.65          # unchanged
NEW_SUBSCORE_DISAGREEMENT_THRESHOLD = 0.45      # was 0.35
EXCLUDE_FUNDAMENTALS_FROM_DISAGREEMENT = True   # structural fix -- see intro
NEW_HUMAN_REVIEW_DISAGREEMENT_THRESHOLD = 0.20  # was 0.30
NEW_HIGH_CRED_SCORE_FLOOR = 0.30                # unchanged -- not covered by calibration data

# --- DISRUPTION ALERTS ---
ALERT_SCORE_THRESHOLD = 0.60   # final_score (tier2) or tier1_composite_score (no tier2) above this alerts


## 1. Recompute the trigger rate under the new logic

Same sweep mechanics as the calibration notebook, but now with `fundamentals` excluded from the
spread and at the new threshold pair -- so you can see, on your real `tier1_composite.csv`,
whether the structural fix actually did what the reasoning above predicted before trusting it.

In [2]:
composite = pd.read_csv(TIER1_COMPOSITE_PATH)
composite["as_of"] = pd.to_datetime(composite["as_of"], utc=True)

OLD_SUB_COLS = ["news_subscore", "vessel_subscore", "fundamentals_subscore"]
NEW_SUB_COLS = ["news_subscore", "vessel_subscore"] if EXCLUDE_FUNDAMENTALS_FROM_DISAGREEMENT else OLD_SUB_COLS

def compute_spread(df, cols):
    spread = df[cols].max(axis=1, skipna=True) - df[cols].min(axis=1, skipna=True)
    n_present = df[cols].notna().sum(axis=1)
    return spread.where(n_present > 1, 0.0)

old_spread = compute_spread(composite, OLD_SUB_COLS)
new_spread = compute_spread(composite, NEW_SUB_COLS)

old_composite_hit = composite["tier1_composite_score"] >= NEW_COMPOSITE_TRIGGER_THRESHOLD
old_triggered = old_composite_hit | (old_spread >= 0.35)   # old config, for comparison
new_triggered = old_composite_hit | (new_spread >= NEW_SUBSCORE_DISAGREEMENT_THRESHOLD)

print(f"OLD config (composite>=0.65 OR fundamentals-inclusive spread>=0.35): "
      f"{old_triggered.mean():.1%} trigger rate ({int(old_triggered.sum()):,} rows)")
print(f"NEW config (composite>=0.65 OR fundamentals-EXCLUDED spread>={NEW_SUBSCORE_DISAGREEMENT_THRESHOLD}): "
      f"{new_triggered.mean():.1%} trigger rate ({int(new_triggered.sum()):,} rows)")
print(f"\nIf the new rate isn't meaningfully lower than the old one, the fundamentals-staleness")
print(f"theory above may not be the actual cause on your real data -- worth re-examining before")
print(f"trusting this config further rather than assuming the reasoning was right.")

composite["tier2_triggered_new"] = new_triggered
composite["subscore_spread_new"] = new_spread


OLD config (composite>=0.65 OR fundamentals-inclusive spread>=0.35): 33.1% trigger rate (3,035 rows)
NEW config (composite>=0.65 OR fundamentals-EXCLUDED spread>=0.45): 1.6% trigger rate (143 rows)

If the new rate isn't meaningfully lower than the old one, the fundamentals-staleness
theory above may not be the actual cause on your real data -- worth re-examining before
trusting this config further rather than assuming the reasoning was right.


## 2. Write the final calibrated config

This is the file to point Step 4's config cell at (or copy the values in by hand) -- either
way, this is the single source of truth for what was actually decided and why.

In [3]:
calibrated = {}
if CALIBRATED_CONFIG_PATH.exists():
    with open(CALIBRATED_CONFIG_PATH) as f:
        calibrated = json.load(f)

final_config = {
    "based_on": str(CALIBRATED_CONFIG_PATH),
    "trigger": {
        "composite_threshold": NEW_COMPOSITE_TRIGGER_THRESHOLD,
        "disagreement_threshold": NEW_SUBSCORE_DISAGREEMENT_THRESHOLD,
        "exclude_fundamentals_from_disagreement": EXCLUDE_FUNDAMENTALS_FROM_DISAGREEMENT,
        "rationale": (
            "Composite threshold unchanged -- sweep showed negligible effect across 0.55-0.75. "
            "Disagreement threshold raised 0.35->0.45 AND fundamentals excluded from the spread "
            "calc -- flat trigger-rate curve across the tested grid pointed to a structural cause "
            "(stale monthly fundamentals mechanically disagreeing with daily news/vessel), not "
            "pure mis-thresholding."
        ),
        "observed_trigger_rate_old_config": float(old_triggered.mean()),
        "observed_trigger_rate_new_config": float(new_triggered.mean()),
    },
    "human_review": {
        "disagreement_threshold": NEW_HUMAN_REVIEW_DISAGREEMENT_THRESHOLD,
        "rationale": (
            "Old flat 0.30 flagged 0/20 real episodes -- the real disagreement_index "
            "distribution topped out at 0.205 (95th percentile), so 0.30 was set above the "
            "ceiling of what the metric could produce. Adopted the calibration notebook's own "
            "80th-percentile suggestion instead."
        ),
    },
    "credibility_floor": {
        "score_floor": NEW_HIGH_CRED_SCORE_FLOOR,
        "rationale": "UNCHANGED -- not covered by the Step 5 calibration notebook. Still an assumption, not a validated value.",
    },
}

with open(FINAL_CALIBRATED_CONFIG, "w") as f:
    json.dump(final_config, f, indent=2, default=str)
print(f"Wrote {FINAL_CALIBRATED_CONFIG}")
print(json.dumps(final_config, indent=2, default=str))

print("\n--- Copy these into Step 4's config cell ---")
print(f"COMPOSITE_TRIGGER_THRESHOLD = {NEW_COMPOSITE_TRIGGER_THRESHOLD}")
print(f"SUBSCORE_DISAGREEMENT_THRESHOLD = {NEW_SUBSCORE_DISAGREEMENT_THRESHOLD}")
print(f"HUMAN_REVIEW_DISAGREEMENT_THRESHOLD = {NEW_HUMAN_REVIEW_DISAGREEMENT_THRESHOLD}")
print(f"HIGH_CRED_SCORE_FLOOR = {NEW_HIGH_CRED_SCORE_FLOOR}  # unchanged")
print("# Also change compute_trigger()'s sub_cols to exclude fundamentals_subscore, per Section 1 above")


Wrote outputs\step5_final_calibrated_config.json
{
  "based_on": "outputs\\step5_calibrated_config.json",
  "trigger": {
    "composite_threshold": 0.65,
    "disagreement_threshold": 0.45,
    "exclude_fundamentals_from_disagreement": true,
    "rationale": "Composite threshold unchanged -- sweep showed negligible effect across 0.55-0.75. Disagreement threshold raised 0.35->0.45 AND fundamentals excluded from the spread calc -- flat trigger-rate curve across the tested grid pointed to a structural cause (stale monthly fundamentals mechanically disagreeing with daily news/vessel), not pure mis-thresholding.",
    "observed_trigger_rate_old_config": 0.33082624809243516,
    "observed_trigger_rate_new_config": 0.015587529976019185
  },
  "human_review": {
    "disagreement_threshold": 0.2,
    "rationale": "Old flat 0.30 flagged 0/20 real episodes -- the real disagreement_index distribution topped out at 0.205 (95th percentile), so 0.30 was set above the ceiling of what the metric could 

---
# Part B — Step 7: Unified Deployment Feed

Merges every Tier 1 row with its Tier 2 result where one exists (a corridor-timestamp inside an
episode that actually got a Hawk/Dove/Adjudicator run), using the **new** calibrated trigger
flag from Part A so the merge reflects the config you're actually shipping, not the old one.

## 3. Load Tier 2 outputs and map them back onto the Tier 1 grid

Tier 2 only scores once per episode (at the episode's end), but its verdict applies to every
Tier 1 grid-point that fell inside that episode's `[episode_start, as_of]` span -- an interval
join, not a point join.

In [4]:
episodes = pd.read_csv(STEP4_EPISODES_PATH)
episodes["as_of"] = pd.to_datetime(episodes["as_of"], utc=True)
episodes["episode_start"] = pd.to_datetime(episodes["episode_start"], utc=True)
episodes["episode_key"] = episodes["corridor_id"] + "::" + episodes["as_of"].astype(str)

with open(STEP4_FINAL_RESULTS_PATH) as f:
    final_results = json.load(f)

print(f"{len(episodes):,} episodes total, {len(final_results):,} have a Tier 2 result "
      f"(the rest were likely beyond MAX_EPISODES_THIS_RUN's cap on the last live run).")

def find_episode_for_row(corridor_id, as_of, episodes_df):
    cand = episodes_df[
        (episodes_df["corridor_id"] == corridor_id)
        & (episodes_df["episode_start"] <= as_of)
        & (episodes_df["as_of"] >= as_of)
    ]
    if cand.empty:
        return None
    return cand.iloc[0]["episode_key"]

# Only attempt the (relatively expensive) interval lookup for rows the NEW trigger flag marks --
# no reason to search for a Tier 2 match on a row that isn't even triggered under the config
# you're shipping.
composite["episode_key"] = None
triggered_idx = composite[composite["tier2_triggered_new"]].index
for i in triggered_idx:
    row = composite.loc[i]
    ek = find_episode_for_row(row["corridor_id"], row["as_of"], episodes)
    composite.loc[i, "episode_key"] = ek

n_matched_to_scored_episode = composite["episode_key"].isin(final_results.keys()).sum()
print(f"{len(triggered_idx):,} rows triggered under the new config, "
      f"{n_matched_to_scored_episode:,} fall inside an episode that has a real Tier 2 result.")


166 episodes total, 20 have a Tier 2 result (the rest were likely beyond MAX_EPISODES_THIS_RUN's cap on the last live run).
143 rows triggered under the new config, 8 fall inside an episode that has a real Tier 2 result.


## 4. Assemble the unified Step 0 schema — one row per corridor-timestamp

Every row gets `tier1_*` fields. Rows matched to a scored episode also get `hawk`/`dove`/
`adjudicator` fields; everything else keeps those null, exactly as the original plan's Step 7
called for -- downstream consumers read one shape regardless of which tier produced it.

In [5]:
def build_unified_row(row):
    ek = row["episode_key"]
    tier2 = final_results.get(ek) if ek else None
    if tier2 is not None:
        hawk, dove, adj = tier2["hawk"], tier2["dove"], tier2["adjudicator"]
        final_score = adj.get("final_score")
    else:
        hawk = dove = adj = None
        final_score = None

    return {
        "corridor_id": row["corridor_id"],
        "as_of": row["as_of"],
        "tier1_composite_score": row["tier1_composite_score"],
        "tier2_triggered": bool(row["tier2_triggered_new"]),
        "tier2_scored": tier2 is not None,
        "hawk_score": hawk.get("score") if hawk else None,
        "dove_score": dove.get("score") if dove else None,
        "adjudicator_final_score": final_score,
        "disagreement_index": adj.get("disagreement_index") if adj else None,
        "human_review_flag": adj.get("human_review_flag") if adj else None,
        "credibility_floor_applied": adj.get("credibility_floor_applied") if adj else None,
        "adjudicator_rationale": adj.get("rationale") if adj else None,
        "effective_score": final_score if final_score is not None else row["tier1_composite_score"],
        "score_source": "tier2_adjudicated" if final_score is not None else "tier1_composite_only",
    }

unified_rows = [build_unified_row(row) for _, row in composite.iterrows()]
unified = pd.DataFrame(unified_rows)
unified.to_csv(UNIFIED_FEED_OUTPUT, index=False)
print(f"Wrote {len(unified):,} unified rows to {UNIFIED_FEED_OUTPUT}")
print(f"\nBreakdown by score_source:")
print(unified["score_source"].value_counts())


Wrote 9,174 unified rows to outputs\unified_risk_feed.csv

Breakdown by score_source:
score_source
tier1_composite_only    9166
tier2_adjudicated          8
Name: count, dtype: int64


## 5. Disruption Alerts export

The concrete deliverable the architecture diagram's "Disruption alerts" box actually needs: a
ranked, human-readable table of corridor-timestamps whose effective score crosses the alert
threshold -- Tier 2's adjudicated score where available, Tier 1's composite alone otherwise.

In [6]:
alerts = unified[unified["effective_score"] >= ALERT_SCORE_THRESHOLD].copy()
alerts = alerts.sort_values(["as_of", "effective_score"], ascending=[False, False])

alerts["lead_line"] = alerts.apply(
    lambda r: (
        f"[{r['corridor_id']}] {r['as_of']} -- score {r['effective_score']:.2f} "
        f"({r['score_source']}"
        + (", HUMAN REVIEW" if r["human_review_flag"] else "")
        + (", credibility floor applied" if r["credibility_floor_applied"] else "")
        + ")"
    ),
    axis=1,
)

print(f"{len(alerts):,} / {len(unified):,} rows cross the alert threshold ({ALERT_SCORE_THRESHOLD})")
print(f"  - via tier2_adjudicated: {(alerts['score_source'] == 'tier2_adjudicated').sum():,}")
print(f"  - via tier1_composite_only: {(alerts['score_source'] == 'tier1_composite_only').sum():,}")
print(f"  - flagged for human review: {alerts['human_review_flag'].sum():,}")

print("\nMost recent alerts:")
for line in alerts["lead_line"].head(15):
    print(" ", line)

alerts.drop(columns=["lead_line"]).to_csv(DISRUPTION_ALERTS_OUTPUT, index=False)
print(f"\nWrote {len(alerts):,} alerts to {DISRUPTION_ALERTS_OUTPUT}")


85 / 9,174 rows cross the alert threshold (0.6)
  - via tier2_adjudicated: 0
  - via tier1_composite_only: 85
  - flagged for human review: 0

Most recent alerts:
  [red_sea] 2026-07-09 06:00:00+00:00 -- score 0.60 (tier1_composite_only)
  [red_sea] 2026-07-09 00:00:00+00:00 -- score 0.61 (tier1_composite_only)
  [red_sea] 2026-07-08 18:00:00+00:00 -- score 0.60 (tier1_composite_only)
  [red_sea] 2026-07-08 12:00:00+00:00 -- score 0.61 (tier1_composite_only)
  [red_sea] 2026-07-08 06:00:00+00:00 -- score 0.62 (tier1_composite_only)
  [red_sea] 2026-07-08 00:00:00+00:00 -- score 0.63 (tier1_composite_only)
  [red_sea] 2026-07-07 18:00:00+00:00 -- score 0.63 (tier1_composite_only)
  [red_sea] 2026-07-07 12:00:00+00:00 -- score 0.64 (tier1_composite_only)
  [red_sea] 2026-07-07 06:00:00+00:00 -- score 0.64 (tier1_composite_only)
  [red_sea] 2026-07-07 00:00:00+00:00 -- score 0.64 (tier1_composite_only)
  [red_sea] 2026-07-06 18:00:00+00:00 -- score 0.64 (tier1_composite_only)
  [red_sea] 

## What this notebook deliberately does NOT do

- **No Knowledge Graph.** A supplier-route-refinery graph needs supplier/refinery/route data
  that doesn't exist anywhere in this project's inputs. Building one on fabricated nodes would
  produce a graph that looks complete and means nothing -- worse than not having one.
- **No Procurement Orchestrator or Reserve Optimisation Agent.** Same reason: both need real
  tanker-availability, port-congestion, and SPR-drawdown data this project never had. The
  `unified_risk_feed.csv` this notebook produces is exactly the input those two agents would
  need if built later -- one score per corridor-timestamp, with full provenance back to which
  tier produced it and why.
- **No live re-run of Step 4 under the new thresholds.** `tier2_triggered_new` in Section 1
  tells you which rows *would* trigger under the calibrated config, but Tier 2 has only
  actually been scored (live LLM calls) for the episodes already in `step4_final_risk_scores.json`
  from your prior run. Newly-triggered rows under the new threshold that fall outside those
  already-scored episodes show up here with `tier2_scored = False` and fall back to
  `tier1_composite_only` until Step 4 is re-run with the calibrated config.